# Step 6: Train, Validation, and Test Splitting

## Purpose
This notebook creates an honest evaluation setup for the Telco churn project.

## Why this step matters
A model can appear strong simply because the evaluation setup is weak. Splitting the data correctly protects us from overestimating model performance.

## Key rule
The test set must stay untouched until the very end. We use training data to fit models and validation data to compare or tune them.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Load the cleaned dataset and reconstruct `X` and `y`

We start from the cleaned dataset so that splitting happens on data that is already consistent.

We reconstruct `X` and `y` here because splitting should be based on the final feature set, not on the raw table.

In [2]:
PROJECT_DIR = Path.cwd()
CLEANED_FILE = PROJECT_DIR / "WA_Fn-UseC_-Telco-Customer-Churn-cleaned.csv"

df = pd.read_csv(CLEANED_FILE)

df['service_count'] = (
    (df['PhoneService'] == 'Yes').astype(int)
    + (df['MultipleLines'] == 'Yes').astype(int)
    + (df['InternetService'] != 'No').astype(int)
    + (df['OnlineSecurity'] == 'Yes').astype(int)
    + (df['OnlineBackup'] == 'Yes').astype(int)
    + (df['DeviceProtection'] == 'Yes').astype(int)
    + (df['TechSupport'] == 'Yes').astype(int)
    + (df['StreamingTV'] == 'Yes').astype(int)
    + (df['StreamingMovies'] == 'Yes').astype(int)
)

df['avg_monthly_value_from_total'] = (df['TotalCharges'] / df['tenure'].replace(0, 1)).round(2)
df['is_new_customer'] = (df['tenure'] <= 12).astype(int)
df['has_long_term_contract'] = df['Contract'].isin(['One year', 'Two year']).astype(int)

X = df.drop(columns=['customerID', 'Churn']).copy()
y = df['Churn'].map({'No': 0, 'Yes': 1}).copy()

X.shape, y.shape

((7032, 23), (7032,))

## 2. Choose the split strategy

For this Telco dataset, **stratified splitting** is the best choice.

## Why not time-based splitting?
A time-based split is best when we have a real time column and want to simulate future prediction. This dataset does not provide a proper event-time structure for that.

## Why not plain random splitting?
A plain random split can accidentally distort the churn ratio between sets, especially when the target is imbalanced.

## Why stratified splitting?
Because churn is a classification problem with an imbalanced target. Stratification keeps the proportion of churners and non-churners similar across training, validation, and test sets. That gives us a fairer evaluation.

## 3. Create train, validation, and test sets

We will split in two stages:

1. First, split off the **test set**.
2. Then split the remaining data into **training** and **validation**.

## Why do it this way?
Because the test set should be isolated as early as possible. Once it is separated, we avoid using it during model choice or tuning.

This setup gives us:
- **60% training**
- **20% validation**
- **20% test**

This is a strong learning setup because it supports both honest evaluation and model comparison.

In [3]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full,
)

print('X_train shape:', X_train.shape)
print('X_val shape:', X_val.shape)
print('X_test shape:', X_test.shape)
print()
print('y_train shape:', y_train.shape)
print('y_val shape:', y_val.shape)
print('y_test shape:', y_test.shape)

X_train shape: (4218, 23)
X_val shape: (1407, 23)
X_test shape: (1407, 23)

y_train shape: (4218,)
y_val shape: (1407,)
y_test shape: (1407,)


In [4]:
split_summary = pd.DataFrame({
    'dataset': ['train', 'validation', 'test'],
    'rows': [len(X_train), len(X_val), len(X_test)],
    'churn_rate': [y_train.mean(), y_val.mean(), y_test.mean()],
})

split_summary['churn_rate'] = split_summary['churn_rate'].round(4)
split_summary

,dataset,rows,churn_rate
0,train,4218,0.2658
1,validation,1407,0.2658
2,test,1407,0.2658


## 4. How each split should be used

- **Training set**: used to fit the model
- **Validation set**: used to compare models, tune hyperparameters, and choose thresholds
- **Test set**: used only once at the end for final evaluation

## Why this discipline matters
If we repeatedly look at the test set while making decisions, it stops being a true test set. Then our final performance estimate becomes optimistic and less trustworthy.

In [5]:
{
    'train_rows': X_train.shape[0],
    'validation_rows': X_val.shape[0],
    'test_rows': X_test.shape[0],
    'total_rows_check': X_train.shape[0] + X_val.shape[0] + X_test.shape[0],
    'original_rows': X.shape[0],
}

{'train_rows': 4218,
 'validation_rows': 1407,
 'test_rows': 1407,
 'total_rows_check': 7032,
 'original_rows': 7032}

# Step 6: Train, Validation, and Test Splitting

## Purpose
This notebook creates an honest evaluation setup for the Telco churn project.

## Why this step matters
A model can appear strong simply because the evaluation setup is weak. Splitting the data correctly protects us from overestimating model performance.

## Key rule
The test set must stay untouched until the very end. We use training data to fit models and validation data to compare or tune them.